Imports and Configs

In [5]:
!pip install pandas
!pip install scikit-learn
!pip install plotly

# !pip install nbformat ipympl # fix for : plot issues

import requests
import os
import time
import pickle 
import math

from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings, Collection
from chromadb.utils.embedding_functions import register_embedding_function

import pandas as pd
from sklearn.decomposition import PCA
import plotly.express as px
# import plotly.io as pio

In [6]:
BASE_URL = "https://api.mangadex.org"
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

Collect Manga without sensitive Tags

In [7]:
def getTagIDs(tags : list[str]) -> list[str]:
    # unordered tag Ids
    ids = []
    response = requests.get(
        f"{BASE_URL}/manga/tag"
    ).json() 
    for row in response["data"]:
        id_  : str = row["id"]
        name : str = row["attributes"]["name"]["en"]
        if(name in tags): ids.append(id_)
    return ids

In [8]:
sensitive_tags = [
    "Loli",
    "Boys' Love",
    "Incest",
    "Sexual Violence",
    "Girls' Love",
    "Harem",
    "Douendinshi",
    "Gore",
    "Horror",
    "Monster Girls",
    "Shota",
    "Tragedy"
]

In [9]:
sensitive_tag_ids = getTagIDs(sensitive_tags)

In [15]:
def getRandomMangas(n : int, exclude_tags : list[str]) -> list[tuple[str, str, str]]:
    # Next version : add genres and other metadata
    # use and throw function - upgrade for api 
    
    mangas = []
    page_per_req = 100
    max_request = (n / float(100)) + 50
    
    request_count = 0
    page_number = 0
    rem = n 
    while(rem > 0 and request_count <= max_request):
        try:
            response = requests.get(
                f"{BASE_URL}/manga",
                params={
                    "excludedTags[]": exclude_tags,
                    "limit" : page_per_req if rem >= page_per_req else rem,
                    "offset" : page_number * page_per_req,
                    "contentRating[]" : ["safe"], 
                    "availableTranslatedLanguage[]" : ["en"]
                },
            ).json()
            request_count += 1
            if(response["result"] != "ok") : 
                print("Retrying for page no :", page_number)
                time.sleep(1)
                continue

            k = 0
            for row in response["data"]:
                # additional checks before insert
                # if not inserted for any reason, recheck logic
                # ...
                
                id_ = row["id"]

                if("en" not in row["attributes"]["description"]): continue
                description = row["attributes"]["description"]["en"]

                title = ""
                if('en' in row["attributes"]["title"]):         title = row["attributes"]["title"]["en"]
                # elif('en' in row["attributes"]["altTitles"]):   title = row["attributes"]["altTitles"]["en"]# Fix this: list of dicts
                elif('enda-ro' in row["attributes"]["title"]):    title = row["attributes"]["title"]["enda-ro"] 
                else:                                           title = "Title for " + id_ 

                mangas.append((id_, description, title))
                k += 1

            page_number += 1
            rem = n - len(mangas)

            time.sleep(1)
        except Exception as e:
            print("Query Failed at : ", page_number)
            print("Reason :", e)
            print("Collected : ", n - rem, "mangas")
            print("Checkpointing: ")
            save_state = dict()
            save_state["mangas"] = mangas
            save_state["page_number"] = page_number
            save_state["n"] = n 
            save_state["rem"] = rem
            with open('saved_manga_state.pkl', 'wb') as file:
                pickle.dump(save_state, file)
            print('saved at :', './saved_manga_state.pkl')
            return mangas 
        
    return mangas 

In [ ]:
# mangas = getRandomMangas(2000, sensitive_tag_ids)

In [ ]:
# with open('saved_mangas.pkl', 'wb') as file:
#     pickle.dump(mangas, file)

In [18]:
with open('saved_mangas.pkl', 'rb') as file:
    mangas = pickle.load(file)

Save to Vector Database (Chroma)

In [20]:
@register_embedding_function
class MistralEmbeddingFunction(EmbeddingFunction):

    def __init__(self):
        self.retry_config = RetryConfig(
            strategy="backoff",
            backoff=BackoffStrategy(
                initial_interval = 1000,
                max_interval = 30_000,  
                exponent = 2, 
                max_elapsed_time = 120_000),
            retry_connection_errors=True
        )
        self.model = Mistral(api_key=MISTRAL_API_KEY, retry_config=self.retry_config) 

    def __call__(self, input: Documents) -> Embeddings:
        res = self.model.embeddings.create(
            model="mistral-embed", 
            inputs=input
        )
        return [x.embedding for x in res.data] 
    
    @staticmethod
    def name() -> str:
        return "Mistral EF (API)"

In [21]:
chroma_client = chromadb.PersistentClient(path="./mangadex_ss")
collection = chroma_client.get_or_create_collection(
    name = "test_collection_1",
    embedding_function = MistralEmbeddingFunction()
)

In [ ]:
# chroma_client.delete_collection("test_collection_1")

In [22]:
# assume token = 4 char
def saveToChromaDB(mangas : list[list[str, str, str]], collection : Collection) -> list[list[str, str, str]]:
    # returns list of failed mangas
    i = 0
    n = len(mangas)
    batch_size = 5 # approx 200 * 5 tokens
    failed_batches = []
    while(i < n):
        end = min(i+batch_size, n)  # exclusive
        b_ids = []
        b_des = []
        b_tit = []

        for id_, des, tit in mangas[i : end]: 
            b_ids.append(id_)
            b_des.append(des)
            b_tit.append(tit)
        try:
            collection.upsert(
                ids=b_ids,
                documents=b_des,
                metadatas=[
                    {"description" : des, "title" : tit}
                    for des, tit in zip(b_des, b_tit)
                ]
            )
        except Exception as e:
            print(f"Failed to upsert : {i}-{end} [{e}]")
            print(f"proceeding to next batch")
            failed_batches.extend(mangas[i:end])
        i = end
    return failed_batches

In [23]:
failed_saves = saveToChromaDB(mangas, collection)

In [24]:
failed_saves

[]

Query

In [25]:
query_result = collection.get(ids=['22d5a76e-de4d-4cf5-8508-d61e978ededa'], include=["embeddings", "metadatas"])
query_result

{'ids': ['22d5a76e-de4d-4cf5-8508-d61e978ededa'],
 'embeddings': array([[-0.05264282,  0.01713562,  0.06390381, ...,  0.00560379,
         -0.02241516, -0.05523682]], shape=(1, 1024)),
 'documents': None,
 'uris': None,
 'included': ['embeddings', 'metadatas'],
 'data': None,
 'metadatas': [{'description': 'Itsumi is a girl who moved to the city with the dream of becoming a cellist. One day, caught by the sound of her music, Warren, a mysterious boy, comes in her room through the window. What does fate have for them after this extraordinary meeting…?',
   'title': 'Tokyo LACKs'}]}

In [26]:
collection.query(
    query_embeddings=query_result["embeddings"],
    n_results=5
)

{'ids': [['22d5a76e-de4d-4cf5-8508-d61e978ededa',
   '23c26ad1-a9d5-4b34-b166-3611a8577e47',
   'eee2d190-668e-4d34-ada0-dffafef066fb',
   '4980c403-32b9-4a8d-b419-baba79ca0b37',
   '8cf0856b-ee23-4f5d-bcb2-e2a604e76f06']],
 'embeddings': None,
 'documents': [['Itsumi is a girl who moved to the city with the dream of becoming a cellist. One day, caught by the sound of her music, Warren, a mysterious boy, comes in her room through the window. What does fate have for them after this extraordinary meeting…?',
   'The story follows the shy and elegant "Idomu" Takaku, who becomes interested in his beautiful classmate Hazuki Shinohara, who sits beside him in class. She seems to keep everyone else at a distance, but the truth is that she has no idea how to communicate with others. Hazuki is secretly interested in Idomu as well, but the two have no idea how to communicate their feelings to each other.',
   'Aoi and Isumi were childhood friends who often played with each other, but before they 

PCA Plot of all embeddings

In [27]:
all_embeddings = collection.get(include=["embeddings", "metadatas"])
len(all_embeddings["ids"])

2000

In [28]:
pca_model = PCA(n_components=3)
pc = pca_model.fit_transform(all_embeddings["embeddings"])

In [32]:
all_embeddings["metadatas"]

[{'title': 'The Skeleton Soldier Failed to Defend the Dungeon',
  'description': 'The Skeleton Soldier is a meager but fiercely loyal fighter who serves to protect its master, Lady Succubus. Its dream of a peaceful life with her is shattered when they\'re both brutally murdered by a group of warriors one day.\n\nBut what would’ve been a pathetic end to an unremarkable soul sparks a new beginning: when the Skeleton Soldier opens its eyes again, it has traveled back 20 years in time!\n\n"I must warn Lady Succubus of all the things to come!"\n\nBut with no special combat skills and a history of failure and defeat, how can it stop the horrible events from unfolding again?\n\nFollow the Skeleton Soldier as it faces the most challenging quest of all — rewriting the ending to its own story.'},
 {'title': 'Title for 2dc7984a-b6dc-493c-aec1-1b33ab7c686a',
  'description': 'A dungeon-guiding gig goes sideways for explorer Stetch Atelier when the prince who hired him tricks him into activating a 

In [ ]:
df = pd.DataFrame()
df["x"] = pc[:, 0]
df["y"] = pc[:, 1]
df["z"] = pc[:, 2]
df["description"] = [d["description"] for d in all_embeddings["metadatas"]]
df["title"] = [d["title"] for d in all_embeddings["metadatas"]]
df

,x,y,z,description,title
0,0.083885,0.184128,0.036399,The Skeleton Soldier is a meager but fiercely ...,The Skeleton Soldier Failed to Defend the Dungeon
1,0.229261,0.091576,-0.050966,A dungeon-guiding gig goes sideways for explor...,Title for 2dc7984a-b6dc-493c-aec1-1b33ab7c686a
2,-0.017400,-0.075395,0.028091,The manga centers on veteran novelist Mariko K...,Title for d579c665-7cdb-4c5f-9370-942231fd35bc
3,0.088091,0.050347,-0.115782,We all know that a lot of the booty of the gol...,C.M.B.
4,-0.139815,-0.121644,0.026095,High school student Mizuno has a person that h...,Title for 4e54f039-7cc2-4748-b814-51336d68e821
...,...,...,...,...,...
1995,0.153027,0.005072,0.125821,"Around the age of thirty, the officer worker S...",Title for b008fe69-1b64-4e2b-820a-ce513979ab17
1996,-0.111704,0.061781,-0.024848,"Hyeun is a genius dancer but has stage fright,...",Title for 2cea9657-a68b-4d21-a3f4-6b74e93082e5
1997,0.037830,0.119495,-0.054973,"In a dream, Riku is being called to a strange ...",Ayuuah
1998,-0.093743,0.047375,0.125023,"The tale follows Shuki, the third princess of ...",Title for b38b0a46-75b8-4a2b-8037-e6f3d5c8fefd


In [42]:
fig = px.scatter_3d(
    data_frame=df,
    x="x",
    y="y",
    z="z",
    hover_data=["title", "description"],
    color="x",
    # text=all_embeddings["ids"],
    labels={'x': 'PC 1', 'y': 'PC 2', 'z': 'PC 3'},
    title='3D PC Graph'
)
fig.show()